# Topic 20b - Stacking with `pd.concat()`

Practice for section 3 of the [main lesson](topic20_data_wrangling.ipynb).

`concat` glues tables together **without matching anything up**. Use it when your tables are the same kind of thing, split into pieces: one file per month, one file per branch, one file per year.

If you need to look values up in another table, you want `merge` instead - that is notebook 20c.

In [ ]:
from pathlib import Path

import pandas as pd


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
print("data folder:", DATA_DIR)

## 1. Stacking rows

The common case: a system exports one file per period and you need them as one table.

In [ ]:
orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])

q1 = orders[orders["order_date"].dt.quarter == 1]
q2 = orders[orders["order_date"].dt.quarter == 2]
q3 = orders[orders["order_date"].dt.quarter == 3]
q4 = orders[orders["order_date"].dt.quarter == 4]

for name, part in [("Q1", q1), ("Q2", q2), ("Q3", q3), ("Q4", q4)]:
    print(f"{name}: {len(part):>4} rows")

year = pd.concat([q1, q2, q3, q4], ignore_index=True)
print(f"\nStacked: {len(year)} rows")
print(f"Matches the original: {len(year) == len(orders)}")

## 2. Why `ignore_index=True`

Without it, each piece keeps its original row labels and the result has a duplicated index.

In [ ]:
without = pd.concat([q1, q2, q3, q4])
with_it = pd.concat([q1, q2, q3, q4], ignore_index=True)

print(f"without ignore_index -> duplicated index: {without.index.duplicated().any()}")
print(f"with ignore_index    -> duplicated index: {with_it.index.duplicated().any()}")
print(f"\nFirst five index labels without: {list(without.index[:5])}")
print(f"First five index labels with:    {list(with_it.index[:5])}")

## 3. The silent failure: mismatched column names

`concat` aligns on **column names**. A name that does not match becomes a new column full of `NaN`, and no error is raised.

In [ ]:
accra = pd.DataFrame({"branch": ["Accra"] * 2, "month": ["Jan", "Feb"],
                      "revenue_ghs": [4200, 3900]})
kumasi = pd.DataFrame({"branch": ["Kumasi"] * 2, "month": ["Jan", "Feb"],
                       "revenue": [3100, 3350]})        # note: different name

broken = pd.concat([accra, kumasi], ignore_index=True)
print(broken.to_string(index=False))
print(f"\nColumns: {list(broken.columns)}")
print(f"Missing values introduced: {broken.isna().sum().sum()}")
print("\nThe total is now wrong in a way nothing warns you about:")
print(f"  sum of revenue_ghs only: {broken['revenue_ghs'].sum()}")
print(f"  true total:              {4200 + 3900 + 3100 + 3350}")

### Your turn - fix it

Rename Kumasi's column before stacking, then confirm the result has three columns, no missing values, and the correct total.

In [ ]:
# YOUR CODE HERE

## 4. Labelling the pieces with `keys=`

When you need to remember which file a row came from.

In [ ]:
labelled = pd.concat([q1, q2, q3, q4],
                     keys=["Q1", "Q2", "Q3", "Q4"],
                     names=["quarter", "row"]).reset_index(level="quarter")
print(labelled[["quarter", "order_id", "order_date"]].head(3).to_string(index=False))
print("\nRows per quarter:")
print(labelled["quarter"].value_counts().sort_index().to_string())

## 5. Stacking columns with `axis=1`

`axis=1` places tables side by side, aligning on the **index**. This is occasionally useful and frequently a trap: if the indexes do not line up, you get `NaN` and misaligned rows rather than an error.

Almost always, if you want to add columns from another table, you want `merge`, not `concat(axis=1)`.

In [ ]:
left = pd.DataFrame({"order_id": ["O1", "O2", "O3"]}, index=[0, 1, 2])
right_aligned = pd.DataFrame({"amount": [10, 20, 30]}, index=[0, 1, 2])
right_shifted = pd.DataFrame({"amount": [10, 20, 30]}, index=[1, 2, 3])

print("Indexes line up:")
print(pd.concat([left, right_aligned], axis=1).to_string(index=False))
print("\nIndexes do NOT line up - four rows appear from three:")
print(pd.concat([left, right_shifted], axis=1).to_string())

## 6. Practice

1. Split `orders` into one table per `channel` using a dictionary comprehension.
2. Stack them back with `keys=` so each row records its channel.
3. Confirm the stacked row count equals the original.
4. Then answer in a markdown cell: why would `merge` have been the wrong tool for this job?

In [ ]:
# YOUR CODE HERE
# by_channel = {name: part for name, part in orders.groupby("channel")}

**Your answer to question 4:**

*(Write your answer here.)*

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: [20a - Tidy data principles](topic20a_tidy_data_principles.ipynb)
- Next: [20c - Merging and joins](topic20c_merging_and_joins.ipynb)